# شام — حارس الدفتر الشامل (يعيد تشغيله فور توقفه)

دفتر صغير بلا مسرّع (CPU، بلا GPU) يُجدوَل بأقصر فاصل يتيحه كاكل. كل تشغيل له يفحص **حالة الدفتر الشامل** فقط: إن كان يعمل لا يفعل شيئاً، وإن توقف (انتهى وقته أو فشل أو أُلغي) دفع له تشغيلاً جديداً، فيبقى الدفتر الشامل يعمل بلا توقف.

هذا المنطق نفسه (`resume_kernel`) الذي في المنسّق القديم، لكن على دفتر واحد معروف لا على كل الدفاتر. لا يدرّب شيئاً بنفسه.

**إعداد لمرة واحدة:**
1. المسرّع **None**، وInternet مفعّل.
2. الأسرار الثلاثة: `GITHUB_TOKEN` و`KAGGLE_USERNAME` و`KAGGLE_KEY`.
3. ضعي في `TARGET_KERNELS` (الخلية التالية) اسم الدفتر الشامل كما يظهر في رابطه: `jonsnowjonsnow/notebookXXXXXXXX`. الدفتر لا يعمل قبل ذلك (توقف مقصود).
4. بعد أن يعمل الحارس، **ألغي جدولة الدفتر الشامل نفسه** (الحارس يشغّله)، كي لا يبدأ اثنان معاً.

**تنبيه موثّق:** دفع دفتر عبر واجهة كاكل قد يفصل أسراره المرفقة ويعيد ضبط مسرّعه حتى لو أبلغ بالنجاح. بعد أول إعادتين تلقائيتين افتحي تبويب Secrets في الدفتر الشامل وتأكدي أنها ما زالت مفعّلة وأن المسرّع كما اخترتِ.


In [ ]:
# ضعي هنا اسم الدفتر الشامل من رابطه (jonsnowjonsnow/notebook...). هذا الدفتر وحده يُراقَب.
TARGET_KERNELS = ["jonsnowjonsnow/REPLACE_ME_ALL_IN_ONE_SLUG"]

assert all(not slug.startswith("REPLACE_ME") and "REPLACE_ME" not in slug for slug in TARGET_KERNELS), (
    "ضعي اسم الدفتر الشامل الحقيقي في TARGET_KERNELS قبل التشغيل (تجده في رابط دفتره)."
)


In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# Same clone/pull pattern as sham_small_training.ipynb's own first cell
# -- this notebook always runs against the LATEST kaggle_auto_resume.py
# in the repo, never a stale copy pasted in here by hand.
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)
print("repo ready, kaggle package installed.")

In [ ]:
import os
import sys
from pathlib import Path as _P
from kaggle_secrets import UserSecretsClient

# Same Kaggle-Secrets pattern already used by every training notebook's
# own checkpoint-publish cell -- no new secret needed, this reuses the
# exact two that already exist for that purpose.
KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

sys.path.insert(0, "/kaggle/working/Ttbik/ai-system/scripts")
from kaggle_auto_resume import _load_kaggle_api, resume_kernel

# One authenticated api object reused across every kernel this run --
# no reason to re-authenticate per target.
api = _load_kaggle_api()

# ذاكرة المنسّق بين التشغيلات (مجموعة بيانات خاصة): أي دفاتر GPU تعمل الآن على CPU مؤقتاً فقط
# لنفاد الرصيد — كي تعود إلى GPU لاحقاً ولا تُعامل كأنك اخترتِ لها CPU.
import json as _json
sys.path.insert(0, "/kaggle/working/Ttbik/ai-system/colab/sham_small")
from sham_inputs import fetch_dataset, publish_dataset
STATE_DATASET = "sham-keeper-state"
_state_dir = fetch_dataset(STATE_DATASET)
_state_files = sorted(_state_dir.rglob("accelerator_state.json")) if _state_dir else []
STATE = _json.loads(_state_files[0].read_text()) if _state_files else {}
_state_before = _json.dumps(STATE, sort_keys=True)

results = {}
for kernel_slug in TARGET_KERNELS:
    print(f"\n=== checking {kernel_slug} ===")
    try:
        # sync_from_repo=False deliberately, same reasoning as the
        # original single-kernel version: a real trained notebook's LIVE
        # Kaggle cells may hold manual edits (calibration constants,
        # hyperparameters) made directly in Kaggle's own editor that are
        # NOT necessarily mirrored back into this git repo's reference
        # copy -- overwriting from the repo here would silently discard
        # those (a real, owner-hit case, 2026-09-21: MAX_TRAINING_HOURS
        # edited live on Kaggle differed from the repo's own value).
        results[kernel_slug] = resume_kernel(
            kernel_slug=kernel_slug,
            notebook_repo_path=None,
            work_dir=f"/kaggle/working/kaggle-kernel/{kernel_slug.replace('/', '_')}",
            sync_from_repo=False,
            api=api,
            auto_accelerator=True,
            state=STATE,
        )
    except Exception as exc:
        print(f"kaggle_auto_resume: unexpected error resuming '{kernel_slug}' ({exc}) -- "
              f"leaving it as-is rather than risking a bad push; check it manually.")
        results[kernel_slug] = f"error: {exc}"

print("\n=== summary ===")
for kernel_slug, outcome in results.items():
    print(f"{kernel_slug}: {outcome}")

if _json.dumps(STATE, sort_keys=True) != _state_before:
    _out = _P("/kaggle/working/orchestrator_state")
    _out.mkdir(parents=True, exist_ok=True)
    (_out / "accelerator_state.json").write_text(_json.dumps(STATE, indent=2))
    publish_dataset(_out, STATE_DATASET, "orchestrator accelerator state")